# Bengali Hate Speech Benchmark: Our Model (BanglaBERT MTL)

Evaluates all 4 experiments (exp1-exp4) on the full 3,553 test set using the **exact same pipeline** as the LLM benchmarks.

**Architecture**: BanglaBERT (110M) + 3 classification heads + optional LSTM decoder
**Metrics**: Macro F1, CVR, Latency


---
## 1. Setup & Load Checkpoints


In [ ]:
!pip install -q transformers accelerate scikit-learn
import os, json, time, glob, gc, re, shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = '/kaggle/working'
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = getattr(torch.cuda.get_device_properties(0), 'total_memory', None)
    if vram is not None:
        print(f'VRAM: {vram / 1e9:.1f} GB')

# Copy model checkpoints from attached dataset (e.g., bangla-hate-trained-models)
search_patterns = [
    '/kaggle/input/**/exp*.pt',
    '/kaggle/input/**/*_best.pt',
    '/kaggle/input/**/*.pt',
    '/kaggle/working/**/exp*.pt',
]
found_models = []
for pat in search_patterns:
    found_models.extend(glob.glob(pat, recursive=True))

found_models = list(dict.fromkeys(found_models))  # deduplicate
if found_models:
    print(f'Found {len(found_models)} candidate model checkpoints:')
    for mf in found_models:
        dest = os.path.join(MODEL_DIR, os.path.basename(mf))
        if not os.path.exists(dest):
            shutil.copy2(mf, dest)
        print(f'  {os.path.basename(mf)} -> {dest}')
else:
    print('WARNING: No model checkpoints found! Please attach bangla-hate-trained-models dataset.')

ENCODER_NAME = 'csebuetnlp/banglabert'
MAX_LENGTH = 256


---
## 2. Load Test Data


In [ ]:
# ── Load Data (same split as NB03/NB04) ──
data_files = glob.glob('/kaggle/input/**/train.json', recursive=True)
test_files = glob.glob('/kaggle/input/**/test.json', recursive=True)

if test_files:
    df_test = pd.read_json(test_files[0])
    print(f'Loaded pre-split test set: {test_files[0]} ({len(df_test)} samples)')
elif data_files:
    print('Only train.json found. Splitting to match NB03/NB04...')
    df_all = pd.read_json(data_files[0])
    df_train_split, df_temp = train_test_split(
        df_all, test_size=0.2, random_state=42,
        stratify=df_all['type_of_hate']
    )
    df_dev, df_test = train_test_split(
        df_temp, test_size=0.5, random_state=42,
        stratify=df_temp['type_of_hate']
    )
    print(f'  Train: {len(df_train_split)}, Val: {len(df_dev)}, Test: {len(df_test)}')
else:
    raise FileNotFoundError('No data found!')

# ── Remap labels ──
_LABEL_REMAP = {'Profane': 'Abusive', 'Sexism': 'Gender Hate'}
df_test['type_of_hate'] = df_test['type_of_hate'].fillna('None').astype(str).str.strip().replace(_LABEL_REMAP)
df_test['target_of_hate'] = df_test['target_of_hate'].fillna('None').astype(str).str.strip()
df_test['severity_of_hate'] = df_test['severity_of_hate'].astype(str).str.strip()

df_eval = df_test.reset_index(drop=True)
print(f'\nEvaluation samples: {len(df_eval)} (FULL test set)')
print(f'Type distribution:\n{df_eval["type_of_hate"].value_counts().to_string()}')


---
## 3. Model Architecture


In [ ]:
# ── Model Architecture (same as training) ──
TYPE_LABELS = ['None', 'Abusive', 'Political Hate', 'Religious Hate', 'Gender Hate']
TARGET_LABELS = ['None', 'Individual', 'Organization', 'Community', 'Society']
SEVERITY_LABELS = ['Little to None', 'Mild', 'Severe']

TYPE2IDX = {label: idx for idx, label in enumerate(TYPE_LABELS)}
TARGET2IDX = {label: idx for idx, label in enumerate(TARGET_LABELS)}
SEVERITY2IDX = {label: idx for idx, label in enumerate(SEVERITY_LABELS)}

class ClassificationHead(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.classifier(x)

class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.projection = nn.Linear(hidden_dim, vocab_size)
        self.init_h = nn.Linear(768, hidden_dim)
        self.init_c = nn.Linear(768, hidden_dim)
        self.num_layers = num_layers
    
    def forward(self, decoder_input_ids, encoder_cls_hidden):
        h0 = self.init_h(encoder_cls_hidden).unsqueeze(0).expand(self.num_layers, -1, -1).contiguous()
        c0 = self.init_c(encoder_cls_hidden).unsqueeze(0).expand(self.num_layers, -1, -1).contiguous()
        embeds = self.embedding(decoder_input_ids)
        lstm_out, _ = self.lstm(embeds, (h0, c0))
        logits = self.projection(lstm_out)
        return logits

class ConsistencyConstrainedMTL(nn.Module):
    def __init__(self, encoder_name='csebuetnlp/banglabert', use_gen_head=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name, attn_implementation="eager")
        enc_dim = self.encoder.config.hidden_size
        self.type_head = ClassificationHead(enc_dim, len(TYPE_LABELS))
        self.target_head = ClassificationHead(enc_dim, len(TARGET_LABELS))
        self.severity_head = ClassificationHead(enc_dim, len(SEVERITY_LABELS))
        self.use_gen_head = use_gen_head
        if use_gen_head:
            vocab_size = self.encoder.config.vocab_size
            self.gen_decoder = LSTMDecoder(vocab_size=vocab_size, embed_dim=256, hidden_dim=512, num_layers=1)
    
    def forward(self, input_ids, attention_mask, decoder_input_ids=None):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_hidden = enc_out.last_hidden_state[:, 0, :]
        type_logits = self.type_head(cls_hidden)
        target_logits = self.target_head(cls_hidden)
        severity_logits = self.severity_head(cls_hidden)
        gen_logits = None
        if self.use_gen_head and decoder_input_ids is not None:
            gen_logits = self.gen_decoder(decoder_input_ids, cls_hidden)
        return type_logits, target_logits, severity_logits, gen_logits

class BanglaHateDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.df = dataframe.copy()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            str(row['comment']), max_length=self.max_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'type_label': torch.tensor(TYPE2IDX.get(row['type_of_hate'], 0), dtype=torch.long),
            'target_label': torch.tensor(TARGET2IDX.get(row['target_of_hate'], 0), dtype=torch.long),
            'severity_label': torch.tensor(SEVERITY2IDX.get(row['severity_of_hate'], 0), dtype=torch.long),
        }

print('Model architecture defined.')


---
## 4. Evaluate All Experiments


In [ ]:
# ── Evaluate All 4 Experiments ──
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
test_dataset = BanglaHateDataset(df_eval, tokenizer, MAX_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def check_consistency_violation(type_idx, target_idx, sev_idx):
    if type_idx == 0:  # None
        if target_idx != 0 or sev_idx != 0:
            return True
    return False

# Mapping of experiments to possible checkpoint filenames
experiments = {
    'exp1_baseline': {
        'name': 'exp1 (Baseline MTL)',
        'files': ['exp1_baseline_best.pt', 'exp1_best.pt', 'exp1.pt']
    },
    'exp2_consistency': {
        'name': 'exp2 (+Consistency Loss)',
        'files': ['exp2_consistency_best.pt', 'exp2_best.pt', 'exp2.pt']
    },
    'exp3_generative': {
        'name': 'exp3 (+Gen Head)',
        'files': ['exp3_generative_best.pt', 'exp3_best.pt', 'exp3.pt']
    },
    'exp4_full': {
        'name': 'exp4 (Full Model)',
        'files': ['exp4_full_best.pt', 'exp4_best.pt', 'exp4.pt']
    },
}

all_results = []

for exp_key, exp_info in experiments.items():
    exp_name = exp_info['name']
    ckpt_path = None
    for fname in exp_info['files']:
        cand1 = os.path.join(MODEL_DIR, fname)
        cand2 = os.path.join('/kaggle/input', fname)
        candidates = [cand1, cand2] + glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        for c in candidates:
            if os.path.exists(c):
                ckpt_path = c
                break
        if ckpt_path:
            break
            
    if not ckpt_path:
        print(f'Skipping {exp_key} — checkpoint not found in {exp_info["files"]}')
        continue
    
    print(f'\n{"="*60}')
    print(f'  Evaluating: {exp_name} from {ckpt_path}')
    print(f'{"="*60}')
    
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    use_gen = checkpoint.get('config', {}).get('use_gen', 'exp3' in exp_key or 'exp4' in exp_key)
    
    model = ConsistencyConstrainedMTL(
        encoder_name=ENCODER_NAME, use_gen_head=use_gen
    ).to(DEVICE)
    
    # Load weights safely
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    model.eval()
    
    preds_type, preds_target, preds_sev = [], [], []
    true_type, true_target, true_sev = [], [], []
    violations = 0
    
    start_time = time.time()
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=exp_name):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            
            type_logits, target_logits, sev_logits, _ = model(input_ids, attention_mask)
            
            type_pred = type_logits.argmax(dim=1).cpu().numpy()
            target_pred = target_logits.argmax(dim=1).cpu().numpy()
            sev_pred = sev_logits.argmax(dim=1).cpu().numpy()
            
            for tp, trp, sp in zip(type_pred, target_pred, sev_pred):
                preds_type.append(TYPE_LABELS[tp])
                preds_target.append(TARGET_LABELS[trp])
                preds_sev.append(SEVERITY_LABELS[sp])
                if check_consistency_violation(tp, trp, sp):
                    violations += 1
            
            true_type.extend([TYPE_LABELS[l] for l in batch['type_label'].numpy()])
            true_target.extend([TARGET_LABELS[l] for l in batch['target_label'].numpy()])
            true_sev.extend([SEVERITY_LABELS[l] for l in batch['severity_label'].numpy()])
    
    elapsed = time.time() - start_time
    latency = elapsed / len(df_eval)
    
    type_f1 = f1_score(true_type, preds_type, average='macro', zero_division=0)
    target_f1 = f1_score(true_target, preds_target, average='macro', zero_division=0)
    sev_f1 = f1_score(true_sev, preds_sev, average='macro', zero_division=0)
    avg_f1 = (type_f1 + target_f1 + sev_f1) / 3
    cvr = violations / len(df_eval) * 100
    
    print(f'  Type F1:       {type_f1:.4f}')
    print(f'  Target F1:     {target_f1:.4f}')
    print(f'  Severity F1:   {sev_f1:.4f}')
    print(f'  Avg Macro F1:  {avg_f1:.4f}')
    print(f'  CVR:           {cvr:.2f}% ({violations}/{len(df_eval)})')
    print(f'  Parse Rate:    100.0% (deterministic)')
    print(f'  Latency:       {latency:.4f}s/sample')
    
    result = {
        'model': exp_name, 'params': '110M',
        'type_f1': round(type_f1, 4), 'target_f1': round(target_f1, 4),
        'sev_f1': round(sev_f1, 4), 'avg_f1': round(avg_f1, 4),
        'cvr_pct': round(cvr, 2), 'violations': violations,
        'total_samples': len(df_eval), 'latency_s': round(latency, 4),
        'parse_rate_pct': 100.0
    }
    all_results.append(result)
    
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Save all results
with open(os.path.join(RESULTS_DIR, 'our_model_benchmark_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2)

# Print summary table
print(f'\n\n{"="*90}')
print(f'  OUR MODEL RESULTS (on same {len(df_eval)} test samples)')
print(f'{"="*90}')
print(f'  {"System":<30} {"Type F1":>8} {"Target F1":>10} {"Sev F1":>8} {"Avg F1":>8} {"CVR":>8} {"Lat(s)":>8}')
print(f'  {"-"*30} {"-"*8} {"-"*10} {"-"*8} {"-"*8} {"-"*8} {"-"*8}')
for r in all_results:
    print(f'  {r["model"]:<30} {r["type_f1"]:>8.4f} {r["target_f1"]:>10.4f} {r["sev_f1"]:>8.4f} {r["avg_f1"]:>8.4f} {r["cvr_pct"]:>7.2f}% {r["latency_s"]:>7.4f}s')
print(f'{"="*90}')
print(f'\n✅ Results saved to {RESULTS_DIR}/our_model_benchmark_results.json')
